In [ ]:
import random

Setup:
- N objects
- N sets with i random objects
- N costs associated to the i sets
- penalty weight > sum of all costs

In [ ]:
N = 100
random.seed(42)

OBJECTS = {n for n in range(N)}
SETS = tuple(frozenset(random.sample(range(N), k=s+1)) for s in range(N))
COSTS = tuple(10*random.random()+(s*random.random())**2 for s in range(N))

PENALTY_WEIGHT = sum(COSTS) + 1.0

In [ ]:
OBJECTS, SETS, COSTS

- The penalty depends on the number of missing objects
- Computing the current state is_valid is crucial to check if the solution is valid

In [ ]:
def fitness(state):
    """
    state: tuple or list of booleans of N elements
    return: fitness value -> (total_cost + penalty) and if it is a valid state
    """
    total_cost = 0.0
    covered = set()

    for i, selected in enumerate(state):
        if selected:
            total_cost += COSTS[i]
            covered.update(SETS[i])

    uncovered_count = len(OBJECTS - covered)
    penalty = uncovered_count * PENALTY_WEIGHT

    return -(total_cost + penalty), uncovered_count == 0

In [ ]:
def tweak(state):
    """
    Flip a random position, based on boolean values
    """
    new_state = list(state)
    flip_i = random.randrange(len(state))
    new_state[flip_i] = not new_state[flip_i]

    return tuple(new_state)

In [ ]:
def hill_climbing(max_evals_without_improvement=200):
    current_state = tuple(random.choice([True, False]) for _ in range(N))
    current_fit, current_valid = fitness(current_state)

    evals = 0
    failed_steps = 0

    while failed_steps < max_evals_without_improvement:
        evals += 1
        new_state = tweak(current_state)
        new_fit, new_valid = fitness(new_state)

        if new_fit > current_fit:
            current_state = new_state
            current_fit = new_fit
            current_valid = new_valid
            failed_steps = 0
            print(evals, current_state, current_fit)
        else:
            failed_steps += 1

    return current_state, current_fit, current_valid, evals

In [ ]:
best_state, best_fit, is_valid, total_evals = hill_climbing()

if is_valid:
    print(f"Best sol: {best_state}")
    print(f"Best fit: {-sum(c for s, c in zip(best_state, COSTS) if s):.4f}")

Last check for best_state dimensionality, we expect d_best_state = N

In [ ]:
covered_elements = set().union(*(s for s, selected in zip(SETS, best_state) if selected))
len(covered_elements)